In [6]:
import numpy as np
import pandas as pd
from scipy.stats import mstats

# ==========================================
# 1. LOAD & PREPARE RAW PRICE MATRIX
# ==========================================
# Read the wide CSV from Phase 1
df_raw = pd.read_csv("market risk model(10).csv")

# Ensure 'Date' is datetime and set as index to align trading days
df_raw["Date"] = pd.to_datetime(df_raw["Date"])
df_raw = df_raw.set_index("Date").sort_index()

# Note: auto_adjust=True in yfinance already adjusted prices for splits and dividends.
prices = df_raw.copy()

# ==========================================
# 2. ALIGN DATES & HANDLE MISSING / STALE VALUES
# ==========================================
# Drop any non-trading rows where all asset prices are NaN
prices = prices.dropna(how="all")

# Forward fill up to 3 consecutive missing values (market holidays, small data gaps)
# followed by backward fill for any edge gaps
prices = prices.ffill(limit=3).bfill()

# Flag & replace stale prices (e.g., flat prices across >= 5 consecutive sessions)
stale_mask = (prices.diff() == 0).rolling(window=5).sum() >= 4
prices[stale_mask] = np.nan
prices = prices.ffill()

# Export: Clean Price Matrix (10)
clean_price_matrix = prices.copy()
clean_price_matrix.to_csv("clean_price_matrix(10).csv")

# ==========================================
# 3. CONVERT PRICES INTO LOG RETURNS
# ==========================================
# r_t = ln(P_t / P_{t-1})
returns = np.log(prices / prices.shift(1)).dropna()

# ==========================================
# 4. WINSORIZE EXTREME RETURN OUTLIERS
# ==========================================
# Cap extreme data artifacts at 0.1% and 99.9% percentiles per ticker
winsorized_returns = returns.apply(
    lambda col: mstats.winsorize(col, limits=[0.001, 0.001])
)

# Export: Clean Return Matrix (10)
clean_return_matrix = pd.DataFrame(
    winsorized_returns, index=returns.index, columns=returns.columns
)
clean_return_matrix.to_csv("clean_return_matrix(10).csv")

# ==========================================
# 5. BUILD WEIGHTED PORTFOLIO RETURNS
# ==========================================
# Equal-weight benchmark across the 10 Dow constituents (10% each)
weights = np.repeat(1.0 / len(clean_return_matrix.columns), len(clean_return_matrix.columns))

# Compute portfolio daily log return series: R_p = sum(w_i * r_i)
portfolio_returns = clean_return_matrix.dot(weights)
portfolio_returns.name = "Portfolio_Log_Return"

# Export: Portfolio Return Series (10)
portfolio_returns.to_csv("portfolio_return_series(10).csv")

# ==========================================
# 6. COMPUTE ROLLING VOLATILITY SERIES
# ==========================================
# 30-day rolling window, annualized assuming 252 trading days/year
# Annualized Vol = rolling_std * sqrt(252)
rolling_window = 30
annualization_factor = np.sqrt(252)

rolling_volatility = (
    portfolio_returns.rolling(window=rolling_window).std() * annualization_factor
)
rolling_volatility.name = "Rolling_30D_Ann_Volatility"

# Export: Rolling Volatility Series (10)
rolling_volatility.dropna().to_csv("rolling_volatility_series(10).csv")

# ==========================================
# SUMMARY VERIFICATION
# ==========================================
print("Phase 2 Execution Completed Successfully.")
print(f"Clean Prices Shape:       {clean_price_matrix.shape}")
print(f"Clean Returns Shape:      {clean_return_matrix.shape}")
print(f"Portfolio Return Count:   {portfolio_returns.shape[0]}")
print(f"Rolling Volatility Count: {rolling_volatility.dropna().shape[0]}")

Phase 2 Execution Completed Successfully.
Clean Prices Shape:       (2512, 10)
Clean Returns Shape:      (2511, 10)
Portfolio Return Count:   2511
Rolling Volatility Count: 2482
